# The Cost of Discretion — Study v2

## A stronger way to study Formula 1 stewarding

Formal FIA decisions do not contain enough common detail to label every ruling fair or unfair.
Study v2 builds the missing structure: a source audit, a public Race Control funnel, incident-lap
windows, close-case matching, and one harm record per driver in a collision.

GPT-5.6 Sol reviewed every included decision and every sampled exclusion under a frozen protocol.
That is a model-led source audit, not independent human annotation. The report is ready as a
reproducible model-reviewed study, but it still withholds claims that need confirmed damage,
counterfactual race effects, or human inter-rater evidence.

In [1]:
# ruff: noqa: E402
import json
import os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".jupyter" / "mplconfig"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display

STRICT = ROOT / "data/manual/study_v2_strict_model_audit/strict-model-audit-0fe15fd6b052"
REVIEW = ROOT / "data/manual/study_v2_review_packets/study-v2-review-7cb1b29b5251"
REFERRAL = ROOT / "data/manual/study_v2_referrals/referrals-a4f9bd038101"
CLOCK = ROOT / "data/manual/study_v2_incident_clock/incident-clock-3dc8bb350308"
CONTEXT = ROOT / "data/manual/study_v2_incident_context/incident-context-707a44aafeb4"
CLOSE = ROOT / "data/manual/study_v2_close_cases/close-cases-b175fe03fa80"
DAMAGE = ROOT / "data/manual/study_v2_damage/damage-screening-23c77a57134e"
LAYERS = ROOT / "data/manual/study_v2_layers/study-v2-layers-eed6774fb6c5"
NATIONALITY = ROOT / "data/manual/study_v2_nationality/nationality-diagnostic-2b1b0ffdd961"
GENERATED = ROOT / "reports/generated/study_v2"
GENERATED.mkdir(parents=True, exist_ok=True)

In [2]:
strict = json.loads((STRICT / "manifest.json").read_text(encoding="utf-8"))
strict_cases = pd.read_csv(STRICT / "strict_model_case_audit.csv", keep_default_na=False)
review = json.loads((REVIEW / "manifest.json").read_text(encoding="utf-8"))
referral = json.loads((REFERRAL / "manifest.json").read_text(encoding="utf-8"))
clock = json.loads((CLOCK / "manifest.json").read_text(encoding="utf-8"))
close = json.loads((CLOSE / "manifest.json").read_text(encoding="utf-8"))
damage = json.loads((DAMAGE / "manifest.json").read_text(encoding="utf-8"))
layers = json.loads((LAYERS / "manifest.json").read_text(encoding="utf-8"))
nationality = json.loads((NATIONALITY / "manifest.json").read_text(encoding="utf-8"))

status = pd.DataFrame([
    {"part": "Strict source audit", "built": f"{strict['included_decisions']} decisions + {strict['exclusion_sources']} exclusions", "release": "Complete; model-led and disclosed"},
    {"part": "Independent human packet", "built": f"{review['reviewer_a_rows']} A / {review['reviewer_b_rows']} B assignments", "release": "Optional future validation; still blank"},
    {"part": "Race Control referral links", "built": f"{referral['high_confidence_link_count']} high-confidence links", "release": "Descriptive"},
    {"part": "Incident clock mapping", "built": f"{clock['mapped_case_count']} of {clock['case_count']} cases", "release": "Validated candidate context"},
    {"part": "Close-case support", "built": f"{close['pre_review_minimum_support_count']} of {close['case_count']} cases", "release": "Review leads only"},
    {"part": "Collision harm records", "built": f"{damage['participant_record_count']} driver records", "release": "Screening only"},
    {"part": "Persistent pace", "built": f"{layers['pace_screen_estimable_rows']} estimable screens", "release": "Waiting for source/context review"},
    {"part": "Proportionality", "built": f"{layers['proportionality_release_rows']} release-ready rows", "release": "Withheld"},
    {"part": "Nationality", "built": f"{nationality['british_accused_rows']} British-accused cases", "release": "Inconclusive"},
])
display(status)
assert strict["records_with_fia_citation"] == strict["unique_sources"] == 920
assert strict["pending_adversarial"] == 0

,part,built,release
0,Strict source audit,418 decisions + 502 exclusions,Complete; model-led and disclosed
1,Independent human packet,496 A / 158 B assignments,Optional future validation; still blank
2,Race Control referral links,177 high-confidence links,Descriptive
3,Incident clock mapping,338 of 346 cases,Validated candidate context
4,Close-case support,317 of 346 cases,Review leads only
5,Collision harm records,412 driver records,Screening only
6,Persistent pace,28 estimable screens,Waiting for source/context review
7,Proportionality,0 release-ready rows,Withheld
8,Nationality,44 British-accused cases,Inconclusive


## 1. What became stronger

**Every reviewed case now has a source.** The strict audit covers 418 included decisions and 502
sampled exclusions. All 920 records cite an exact FIA URL. It confirmed 884 records, corrected 32,
and left four unavailable archive labels visibly unresolved. It never fills the blank human-review
ledgers or calls the same model an independent reviewer.

**The audit changed data, not just wording.** Seven decisions had fault-language errors. Twenty-five
had a missing affected-driver list that the cited source could resolve. Affected-driver coverage
rose from 372 of 418 decisions to 397 of 418. The remaining 21 stay blank because the public source
does not identify a driver clearly enough.

**The population boundary is more visible.** The public timing feed contains 966 Race Control
episodes. Of 346 formal primary decisions, 177 link to an episode at high confidence. Candidate and
ambiguous links remain visible rather than being forced into the analysis.

**Incident timing is much better.** FIA local incident clocks map 338 of 346 cases into lap windows.
The method reproduces all 31 cases that already had a known lap. It gives 174 single-lap candidates;
wider windows remain uncertain.

**Similar cases are compared without looking at the result.** The matching process excludes fault,
penalty, damage, retirement, and finish. It finds at least five neighbors for 317 cases. A different
outcome inside a close pair is a reason to read both sources, not a finding that either ruling was
wrong.

**Harm now follows every participant.** The 233 collision decision rows reduce to 193 candidate
incidents and expand to 412 driver-level harm records. This handles chain collisions and different
types of harm to different drivers.

In [3]:
audit_status = (
    strict_cases.groupby(["review_scope", "strict_model_review_status"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
corrections = strict_cases.loc[
    strict_cases["model_correction_fields"].ne(""),
    ["document_id", "event_name", "title", "model_correction_fields", "model_correction_rationale", "fia_decision_citation_url"],
]
display(audit_status)
display(corrections)
display(Markdown("[Download the full 920-row source-cited audit](../data/manual/study_v2_strict_model_audit/strict-model-audit-0fe15fd6b052/strict_model_case_audit.csv)"))

,review_scope,strict_model_review_status,records
0,exclusion_qa,model_confirmed_from_cited_source,498
1,exclusion_qa,model_unresolved_public_evidence,4
2,primary,model_confirmed_from_cited_source,314
3,primary,model_corrected_from_cited_source,32
4,secondary,model_confirmed_from_cited_source,72


,document_id,event_name,title,model_correction_fields,model_correction_rationale,fia_decision_citation_url
504,fia-2018-abu-b67f80da207e,Abu Dhabi Grand Prix,Stewards Decision Doc35 - N.Hülkenberg,affected_driver_numbers,The cited FIA fact identifies Car 8 as the cou...,https://www.fia.com/sites/default/files/doc_35...
537,fia-2018-jpn-81677672dab7,Japanese Grand Prix,Stewards Decision Doc38 - S.Vettel,affected_driver_numbers,The cited FIA fact identifies Car 33 as the co...,https://www.fia.com/sites/default/files/doc_38...
546,fia-2018-sgp-21e75b934044,Singapore Grand Prix,Stewards Decision Doc34 - K.Magnussen,fault_language,The source explicitly rejects predominant faul...,https://www.fia.com/sites/default/files/doc_34...
583,fia-2019-jpn-9be40f8bba4d,Japanese Grand Prix,Decision - Car 33 (turn 1 incident with car 16),fault_language,The decision is addressed to Car 33; Car 16's ...,https://www.fia.com/sites/default/files/decisi...
599,fia-2020-70a-2f908d545c9e,70th Anniversary Grand Prix,Offence - Car 20 - Re-joined the track in an u...,affected_driver_numbers,Latifi drove Car 6 at this event; the FIA reas...,https://www.fia.com/sites/default/files/decisi...
605,fia-2020-esp-bc594cf0008c,Spanish Grand Prix,Decision - Car 99 - Allegedly forcing another ...,affected_driver_numbers,Grosjean drove Car 8; the FIA reason identifie...,https://www.fia.com/sites/default/files/decisi...
617,fia-2021-aut-01e04f2a38ce,Austrian Grand Prix,Offence - Car 11 - forcing another driver off ...,affected_driver_numbers,Leclerc drove Car 16; the FIA reason identifie...,https://www.fia.com/sites/default/files/decisi...
619,fia-2021-aut-5c20f18ec25e,Austrian Grand Prix,Offence - Car 11 - forcing another driver off ...,affected_driver_numbers,Leclerc drove Car 16; the FIA reason identifie...,https://www.fia.com/sites/default/files/decisi...
621,fia-2021-aut-f412d61f822f,Austrian Grand Prix,Offence - Car 4 - forcing another driver off t...,affected_driver_numbers,Perez drove Car 11; the FIA reason identifies ...,https://www.fia.com/sites/default/files/decisi...
628,fia-2021-gbr-74d32cf58767,British Grand Prix,Decision - Car 11 - Turn 17 incident with car ...,fault_language,The source explicitly rejects predominant faul...,https://www.fia.com/sites/default/files/decisi...


[Download the full 920-row source-cited audit](../data/manual/study_v2_strict_model_audit/strict-model-audit-0fe15fd6b052/strict_model_case_audit.csv)

## 2. Damage evidence and pace loss

No single public database reliably records Formula 1 damage. The collection method therefore joins
FIA timing and classifications with official team reports, named driver or engineer accounts, and
Formula1.com reporting. Team accounts can identify a floor, wing, puncture, repair, or attributed
pace cost, but they are interested-party evidence and must be checked against official timing.

Driver-specific clock mapping gives a single incident lap for 241 harm records. Fifty-two have the
minimum clean laps before and after plus teammate coverage. Exact same-lap matching leaves 28
estimable timing screens. These are not confirmed damage effects: tyre choice, traffic, strategy,
weather, and hidden car conditions can still drive the result.

The source method is documented with official examples, including
[Hamilton's attributed Imola front-wing loss](https://www.formula1.com/en/latest/article/front-wing-damage-cost-hamilton-0-6s-per-lap-until-imola-red-flag-mercedes.4YdB5ZdPJaoMCfjnx5Nk3u),
[Piastri's Miami wing change](https://www.formula1.com/en/latest/article/sainz-hit-with-five-second-time-penalty-after-collision-with-piastri-in.3D1JHk6lYz0GzKch77GcrZ), and
[Williams' description of worsening floor damage in Japan](https://www.williamsf1.com/posts/05f49fb5-62ac-4308-b14d-52f8959cfee8/2023-japanese-grand-prix).

In [4]:
display(Markdown("![Referral funnel](../reports/generated/study_v2/referral_funnel.png)"))
display(Markdown("![Pace screens](../reports/generated/study_v2/pace_screen_distribution.png)"))
display(Markdown("![Nationality power](../reports/generated/study_v2/nationality_power_v2.png)"))

![Referral funnel](../reports/generated/study_v2/referral_funnel.png)

![Pace screens](../reports/generated/study_v2/pace_screen_distribution.png)

![Nationality power](../reports/generated/study_v2/nationality_power_v2.png)

## 3. Conduct, harm, and punishment stay separate

Study v2 does not create one fairness score. It stores:

1. the act and the written finding;
2. each participant's observed consequence;
3. the nominal and realized cost of the sanction, in seconds, positions, grid places, or points.

A proportionality comparison needs three different things: a fault finding, source-supported harm,
and the realized cost of the sanction. The strict audit now covers the first part. Damage and actual
sanction cost are still incomplete, so no full-corpus record meets every gate and the release count
remains zero. This is a boundary on the claim, not a failed analysis.

The FIA's public 2025 guideline explanation says the guidelines assist steward decisions but are
not regulations. Historical FIA practice was also described as judging the incident rather than its
outcome. Damage therefore measures consequence; it does not back-fill fault.

## 4. Nationality remains inconclusive

British accused drivers received sanctions in 25 of 44 formal cases (56.8%). Other accused drivers
received sanctions in 189 of 302 cases (62.6%). This raw 5.8-point difference is not an adjusted
effect and does not show favoritism.

Measured overlap passes the frozen balance checks, but the British group is below the required 98
cases. Simulated power for the prespecified 15-point difference is only 37.8% to 53.6%, depending on
the baseline rate. The model-led source audit is complete, but independent human validation is not.
More importantly, the sample-size and power gates fail on their own. No adjusted nationality result
is fit or released.

## 5. What is settled, and what is still open

The user does not need to work through hundreds of FIA decisions. The model-led audit is complete,
source-cited, and used by the downstream analysis. The blank Reviewer A and Reviewer B files remain
available only if a future reviewer wants to measure independent agreement.

The open work is narrower:

- confirm damage, repair, retirement, and rare benefit claims with incident-specific sources;
- verify the highest-priority close pairs with public video when a reliable clip exists;
- record when and where each sanction was served before calling nominal seconds an actual race cost;
- collect more seasons or cases before testing a small nationality effect.

Until those gates pass, the defensible conclusion is narrow: formal FIA decisions can be audited for
coding consistency, but the public record still cannot support a population-wide verdict that
stewarding is fair, unfair, biased, or proportional to race harm.

## Evidence status

| Output | Current status |
|---|---|
| Strict source audit | 920 of 920 cited; 32 included rows corrected; model-led disclosure |
| Primary 346-case population | Rebuilt from strict reviewed fields |
| Referral funnel | Descriptive public-feed coverage |
| Incident lap windows | Validated candidate context |
| Close-case neighbors | Review-priority tool |
| Damage and pace screens | Source-research tool |
| Proportionality | Withheld pending damage and realized-sanction evidence |
| Nationality effect | Inconclusive; release gate failed |

The protocol, source hierarchy, packets, transformations, and release gates are versioned in the
repository. Unknowns remain unknown instead of being converted into zeros.

## Appendix: one FIA citation for every included decision

The table below contains all 418 included decisions. Each link goes directly to that decision's FIA
source. The downloadable 920-row audit also includes the 502 exclusion checks, rule sources,
evidence spans, corrections, and unresolved-source status.

In [5]:
decision_citations = strict_cases.loc[
    strict_cases["review_scope"].isin(["primary", "secondary"]),
    ["season", "event_name", "review_scope", "title", "document_id", "fia_decision_citation_url"],
].copy()
decision_citations["FIA decision"] = decision_citations["fia_decision_citation_url"].map(
    lambda url: f'<a href="{url}">Official source</a>'
)
decision_citations = decision_citations.drop(columns="fia_decision_citation_url")
assert len(decision_citations) == 418
display(HTML(decision_citations.to_html(index=False, escape=False)))

season,event_name,review_scope,title,document_id,FIA decision
2018,Abu Dhabi Grand Prix,primary,Offence Doc38 - F.Alonso,fia-2018-abu-18bc30990ea3,Official source
2018,Abu Dhabi Grand Prix,primary,Offence Doc39 - F.Alonso,fia-2018-abu-754824eaae10,Official source
2018,Abu Dhabi Grand Prix,primary,Stewards Decision Doc35 - N.Hülkenberg,fia-2018-abu-b67f80da207e,Official source
2018,Abu Dhabi Grand Prix,primary,Offence Doc37- F.Alonso,fia-2018-abu-ba74bbf1e4a2,Official source
2018,Abu Dhabi Grand Prix,primary,Offence Doc36 - E.Ocon,fia-2018-abu-e3bd8959c0f4,Official source
2018,Austrian Grand Prix,primary,Offence Doc42- K.Räikkönen,fia-2018-aut-6376daf466a5,Official source
2018,Austrian Grand Prix,primary,Offence Doc43 - C.Sainz,fia-2018-aut-9c50973f6716,Official source
2018,Azerbaijan Grand Prix,primary,Stewards Decision Doc46 - M.Verstappen,fia-2018-aze-1bbf84530187,Official source
2018,Azerbaijan Grand Prix,primary,Stewards Decision Doc33 - M.Ericsson,fia-2018-aze-23f8b0b5b715,Official source
2018,Azerbaijan Grand Prix,primary,Stewards Decision Doc31 - E.Ocon,fia-2018-aze-3e59a6773794,Official source
